# Competitor Gap Analysis — Rising Tide
### Rising Tide — Local Embeddings + K-Means Clustering

Sam Torres — The SEO Mermaid

---

## Who this notebook is for
You have a Screaming Frog license (or any crawl export) and nothing else. No API keys, no budget, no data leaving your machine. Everything in this notebook runs locally and costs nothing.

## What you'll get
- A topic cluster map of your site vs. a competitor's
- Breadth and depth charts showing where the gaps are
- A CSV of every page with its cluster assignment
- A UMAP visualization of the topic landscape

## What you need
- A CSV export of your site (Screaming Frog or custom)
- A CSV export of your competitor's site
- That's it

## Quick start
1. Upload your CSVs
2. Fill in the CONFIGURATION cell
3. Runtime > Run all


## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy matplotlib seaborn umap-learn -q
print("Dependencies installed")

## Configuration — edit this cell before running

In [ ]:
# FILE PATHS
YOUR_SITE_CSV  = "your_site.csv"
COMPETITOR_CSV = "competitor_site.csv"

# COLUMN MAPPING
# Screaming Frog defaults:
URL_COLUMN   = "Address"
TEXT_COLUMNS = ["Title 1", "Meta Description 1", "H1-1"]
# Custom CSV? Change these to match your headers.

# LABELS
YOUR_SITE_LABEL  = "My Site"
COMPETITOR_LABEL = "Competitor"

# CLUSTERING
# Rule of thumb: sqrt(total_pages / 2). Adjust after first run.
N_CLUSTERS = 10

# EMBEDDING MODEL
ST_MODEL_NAME = "all-MiniLM-L6-v2"  # Fast and free. Alternative: "all-mpnet-base-v2"

print("Configuration loaded")

## Step 1: Load and prepare data
We load both CSVs, filter to live HTML pages, and combine text columns into a single content field per page.

In [ ]:
import pandas as pd
import numpy as np

def load_and_prep(filepath, url_col, text_cols, label):
    df = pd.read_csv(filepath, low_memory=False)
    df = df[df[url_col].notna() & (df[url_col].str.strip() != "")].copy()
    if "Content Type" in df.columns:
        df = df[df["Content Type"].str.contains("text/html", na=False)]
    if "Status Code" in df.columns:
        df = df[df["Status Code"] == 200]
    available_cols = [c for c in text_cols if c in df.columns]
    if not available_cols:
        raise ValueError(f"None of {text_cols} found in {filepath}. Check TEXT_COLUMNS config.")
    df["content"] = df[available_cols].fillna("").apply(
        lambda row: " | ".join(v for v in row if str(v).strip()), axis=1
    )
    df = df[df["content"].str.strip() != ""].copy()
    df = df[[url_col, "content"]].rename(columns={url_col: "url"})
    df["source"] = label
    df = df.reset_index(drop=True)
    print(f"  {label}: {len(df)} pages loaded")
    return df

print("Loading data...")
df_yours = load_and_prep(YOUR_SITE_CSV,  URL_COLUMN, TEXT_COLUMNS, YOUR_SITE_LABEL)
df_comp  = load_and_prep(COMPETITOR_CSV, URL_COLUMN, TEXT_COLUMNS, COMPETITOR_LABEL)
df_all   = pd.concat([df_yours, df_comp], ignore_index=True)
print(f"\nTotal pages to embed: {len(df_all)}")
df_all.head()

## Step 2: Generate embeddings
We convert each page's content into a semantic vector using Sentence Transformers.
This runs entirely on your machine — no API key, no cost, no data sent anywhere.

In [ ]:
def get_embeddings_st(texts, model_name):
    from sentence_transformers import SentenceTransformer
    print(f"Loading Sentence Transformers model: {model_name}")
    print("(First run downloads ~90MB — cached after that)")
    model = SentenceTransformer(model_name)
    print(f"Embedding {len(texts)} pages...")
    return model.encode(texts, show_progress_bar=True, batch_size=64)
# TIP: Save embeddings to disk to avoid re-running this step
# import numpy as np
# np.save("embeddings.npy", embeddings)
# To reload: embeddings = np.load("embeddings.npy")

texts      = df_all["content"].tolist()
embeddings = get_embeddings_st(texts, ST_MODEL_NAME)
print(f"\nEmbeddings shape: {embeddings.shape}")

## Step 3: Cluster into topic neighborhoods
K-means groups pages into topic neighborhoods. Both your pages and your competitor's
are clustered together, so you can see where they have coverage you don't.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

embeddings_norm = normalize(embeddings)
print(f"Clustering {len(df_all)} pages into {N_CLUSTERS} clusters...")

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, max_iter=300)
df_all["cluster"] = kmeans.fit_predict(embeddings_norm)

# Simple numeric labels for k-means
topic_name_map = {i: f"Cluster {i}" for i in range(N_CLUSTERS)}
df_all["topic_name"] = df_all["cluster"].map(topic_name_map)

cluster_summary = (
    df_all.groupby(["cluster", "source"])
    .size().unstack(fill_value=0).reset_index()
)
for col in [YOUR_SITE_LABEL, COMPETITOR_LABEL]:
    if col not in cluster_summary.columns:
        cluster_summary[col] = 0

cluster_summary["total"]     = cluster_summary[YOUR_SITE_LABEL] + cluster_summary[COMPETITOR_LABEL]
cluster_summary["gap_ratio"] = (
    cluster_summary[COMPETITOR_LABEL] / cluster_summary[YOUR_SITE_LABEL].replace(0, 0.1)
).round(2)
cluster_summary = cluster_summary.sort_values("gap_ratio", ascending=False)

print("\nCluster summary (highest competitor advantage first):")
print(cluster_summary[["cluster", YOUR_SITE_LABEL, COMPETITOR_LABEL, "total", "gap_ratio"]].to_string(index=False))

## Step 4: Visualize — Breadth & Depth charts
Two charts: page counts per cluster (depth) and a normalized coverage heatmap (breadth).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Competitor Gap Analysis: Breadth & Depth", fontsize=16, fontweight="bold", y=1.02)

cs = cluster_summary.copy()
cs["label"] = cs["cluster"].astype(str)
x, w = range(len(cs)), 0.35

ax1 = axes[0]
ax1.bar([i - w/2 for i in x], cs[YOUR_SITE_LABEL],  w, label=YOUR_SITE_LABEL,  color="#028090", alpha=0.85)
ax1.bar([i + w/2 for i in x], cs[COMPETITOR_LABEL], w, label=COMPETITOR_LABEL, color="#F96167", alpha=0.85)
ax1.set_xticks(list(x))
ax1.set_xticklabels(cs["label"], rotation=45, ha="right", fontsize=9)
ax1.set_ylabel("Number of Pages")
ax1.set_title("Depth: Pages per Cluster")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

ax2 = axes[1]
hm = cs[[YOUR_SITE_LABEL, COMPETITOR_LABEL]].copy()
hm.index = cs["label"]
sns.heatmap(hm.div(hm.max()).fillna(0).T, ax=ax2, cmap="YlOrRd",
            linewidths=0.5, annot=hm.T, fmt="g", annot_kws={"size": 8},
            cbar_kws={"label": "Relative coverage"})
ax2.set_title("Breadth: Coverage Heatmap")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.savefig("competitor_gap_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: competitor_gap_analysis.png")

## Step 5: Topic space map (UMAP)

In [ ]:
from umap import UMAP
import matplotlib.pyplot as plt

print("Running 2D UMAP for visualization (30-60s)...")
reducer_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer_2d.fit_transform(embeddings)
df_all["umap_x"] = coords[:, 0]
df_all["umap_y"] = coords[:, 1]

fig, ax = plt.subplots(figsize=(13, 8))
site_colors  = {YOUR_SITE_LABEL: "#028090", COMPETITOR_LABEL: "#F96167"}
site_markers = {YOUR_SITE_LABEL: "o", COMPETITOR_LABEL: "^"}

outliers = df_all[df_all["cluster"] == -1] if -1 in df_all["cluster"].values else pd.DataFrame()
if len(outliers) > 0:
    ax.scatter(outliers["umap_x"], outliers["umap_y"],
               c="#CCCCCC", alpha=0.3, s=20, zorder=1, label="Outlier")

for source, group in df_all[df_all["cluster"] != -1].groupby("source"):
    ax.scatter(group["umap_x"], group["umap_y"],
               c=site_colors.get(source, "#888888"), marker=site_markers.get(source, "o"),
               alpha=0.65, s=45, label=source, zorder=2)

for cid, group in df_all[df_all["cluster"] != -1].groupby("cluster"):
    cx, cy = group["umap_x"].mean(), group["umap_y"].mean()
    label = str(topic_name_map.get(cid, f"T{cid}"))[:22]
    ax.annotate(label, (cx, cy), fontsize=8, fontweight="bold", color="#0D1B2A",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, ec="#CCCCCC"))

ax.set_title("Topic Space Map: Your Site vs. Competitor", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP dimension 1")
ax.set_ylabel("UMAP dimension 2")
ax.legend(markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("topic_space_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: topic_space_map.png")


## Step 6: Sample pages per cluster
Review what's actually in each cluster to sanity-check the groupings.

In [ ]:
SAMPLE_PER_CLUSTER = 5

for cluster_id in sorted(df_all["cluster"].unique()):
    cluster_pages = df_all[df_all["cluster"] == cluster_id]
    sample = cluster_pages.sample(min(SAMPLE_PER_CLUSTER, len(cluster_pages)), random_state=42)
    your_count = len(cluster_pages[cluster_pages["source"] == YOUR_SITE_LABEL])
    comp_count = len(cluster_pages[cluster_pages["source"] == COMPETITOR_LABEL])

    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id} | {YOUR_SITE_LABEL}: {your_count} | {COMPETITOR_LABEL}: {comp_count} | gap_ratio: {comp_count / max(your_count, 0.1):.1f}x")
    print(f"{'='*60}")
    for _, row in sample.iterrows():
        print(f"  [{row['source']}] {row['url']}")
        print(f"    {row['content'][:100]}...")

## Step 7: Export

In [ ]:
df_all[["url", "source", "cluster", "topic_name", "content"]].to_csv("all_pages_clustered.csv", index=False)
cluster_summary.to_csv("competitor_gap_summary.csv", index=False)

print(f"Exported {len(df_all)} pages to all_pages_clustered.csv")
print("Exported cluster summary to competitor_gap_summary.csv")
print("\nNext step: review the cluster samples above, label the clusters manually,")
print("and use the gap_ratio column to prioritize content opportunities.")

---
## You're done!

### Output files
| File | What it contains |
|------|-----------------|
| `competitor_gap_analysis.png` | Breadth & depth charts |
| `topic_space_map.png` | UMAP topic landscape |
| `competitor_gap_summary.csv` | Cluster counts and gap ratios |
| `all_pages_clustered.csv` | Every page with cluster assignment |

### Tuning
- Too many small clusters? Increase `N_CLUSTERS`
- Too few broad clusters? Decrease `N_CLUSTERS`
- Re-run from Step 3 only — no need to re-embed

### Ready to go further?
When you have an API key and $10-20 to work with, move to **Open Water** to add automatic cluster labels and a prioritized gap report.

---
*Sam Torres — The SEO Mermaid*
